# 3. Add a GPU w/ Read Only 

This notebook starts a GPU accelerated Presto stack.  
The Presto native worker runs [cuDF](https://github.com/rapidsai/cudf) (CUDA-X for Data Frames).  
Presto + GPU stack shares `metastore/` and `warehouse/` with CPU stack, but limited to read-only.

The images: 

 -  prestodb/presto:`coordinator-gpu-nightly`
 -  prestodb/presto-native:`gpu-nightly`

The connection ports:

 - CPU Presto: port **8080** 
 - GPU Presto: port **8090**.

## 3.1 GPU configs

1. Clone the warehouse configs
2. Add the cuDF settings to the worker (the only config difference between CPU & GPU)

### Clone Configs

In [1]:
# remove old if existing
!rm -rf etc_GPU_coordinator etc_GPU_worker

# make copies
!cp -r  etc_CPU_coordinator etc_GPU_coordinator
!cp -r  etc_CPU_worker      etc_GPU_worker

### Add GPU Settings

In [2]:
def append_if_missing(path, content, marker):
    if marker not in open(path).read():
        with open(path, "a") as f:
            f.write(content)

append_if_missing(
    "etc_GPU_worker/config.properties",
    """
# cuDF: run native operators on the GPU
cudf.enabled=true
cudf.memory_resource=async
cudf.exchange=false
cudf.jit_expression_enabled=false
cudf.intra_node_exchange=true
cudf.partitioned_output_batch_rows=100000000
cudf.concat_optimization_enabled=true
cudf.batch_size_min_threshold=100000000
cudf.batch-splits-enabled=true
cudf.max-files-per-batch-split=0
sink.max-buffer-size=8GB
task.max-drivers-per-task=12
""",
    marker="cudf.enabled",
)

append_if_missing(
    "etc_GPU_worker/catalog/hive.properties",
    """
# cuDF Parquet reader
cudf.hive.use-buffered-input=true
cudf.hive.use-experimental-reader=false
parquet.reader.chunk-read-limit=2147483648
parquet.reader.pass-read-limit=17179869184
""",
    marker="cudf.hive.use-buffered-input",
)

## 3.2 Start GPU Presto

In [3]:
!docker compose -f docker-compose-presto-GPU.yaml up -d

# shutdown:
# docker compose -f docker-compose-presto-GPU.yaml down

In [4]:
!docker ps -a

CONTAINER ID   IMAGE                                     COMMAND                CREATED         STATUS                   PORTS                                         NAMES
dbb22833e630   prestodb/presto-native:gpu-nightly        "/opt/entrypoint.sh"   9 seconds ago   Up 8 seconds (healthy)   0.0.0.0:8092->8082/tcp, [::]:8092->8082/tcp   worker-gpu
fb500d131b71   prestodb/presto:coordinator-gpu-nightly   "/opt/entrypoint.sh"   9 seconds ago   Up 8 seconds (healthy)   0.0.0.0:8090->8080/tcp, [::]:8090->8080/tcp   coordinator-gpu
57a519eccf63   prestodb/presto-native:latest             "/opt/entrypoint.sh"   2 minutes ago   Up 2 minutes (healthy)   0.0.0.0:8082->8082/tcp, [::]:8082->8082/tcp   worker-cpu
2bc7095190ab   prestodb/presto:latest                    "/opt/entrypoint.sh"   2 minutes ago   Up 2 minutes (healthy)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp   coordinator-cpu


In [5]:
# smoke test
import prestodb

conn = prestodb.dbapi.connect(host="localhost", port=8090, user="nb", catalog="hive", schema="aml_hi_lg")
cur = conn.cursor()
cur.execute(" SELECT * FROM transactions LIMIT 1 ")
print(cur.fetchall())
conn.close()

[[275244, '8369AE2D0', 220255, '841DE3900', 235.91, 'US Dollar', 235.91, 'US Dollar', 'Credit Card', 0, '2022-09-23 02:23:00.000', '2022-09-23']]


## 3.3 Same query, CPU vs GPU

In [6]:
import prestodb

conns = {}
curs = {}

for engine, port in [("CPU", 8080), ("GPU", 8090)]:
    print(engine,port)
    conn = prestodb.dbapi.connect(host="localhost", 
                                  port=port, 
                                  user="nb", 
                                  catalog="hive", 
                                  schema="aml_hi_lg")
    conns[engine] = conn
    curs[engine] = conn.cursor()

CPU 8080
GPU 8090


In [7]:
import time

sql = """
SELECT
    from_bank,
    from_account,
    payment_currency,
    COUNT(*) AS txn_count,
    SUM(amount_paid) AS total_paid,
    AVG(amount_paid) AS avg_paid,
    MAX(amount_paid) AS max_paid
FROM 
    transactions
GROUP BY 
    from_bank, 
    from_account, 
    payment_currency
ORDER BY 
total_paid DESC
LIMIT 10
"""

results = {}
for cur in curs.items():
    print(cur[0], end=" ->\t")
    
    start_exec = time.perf_counter()

    cur[1].execute(sql)
    
    exec_time = time.perf_counter() - start_exec
    start_fetch = time.perf_counter()
    
    results[cur[0]] = cur[1].fetchall()
    
    fetch_time = time.perf_counter() - start_fetch
    total_time = exec_time + fetch_time

    print(round(total_time,2), " =", round(exec_time,2), " + ", round(fetch_time,2))

CPU ->	5.74  = 5.74  +  0.0
GPU ->	1.6  = 1.6  +  0.0


In [8]:
# close curs and conns
for conn in conns.items():
    print(conn[0])
    conn[1].close()

CPU
GPU


## 3.4 Shutdown Docker

In [9]:
!docker compose -f docker-compose-presto-native.yaml down

In [10]:
!docker compose -f docker-compose-presto-GPU.yaml down

## 3.5 Docker Compose Diff

In [11]:
import difflib
from IPython.display import HTML

a, b = [open(f).readlines() for f in ("docker-compose-presto-native.yaml", "docker-compose-presto-GPU.yaml")]

HTML(difflib.HtmlDiff(wrapcolumn=70).make_file(a, b, "CPU", "GPU", context=True))